In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/march-machine-learning-mania-2026/Conferences.csv
/kaggle/input/march-machine-learning-mania-2026/WNCAATourneyDetailedResults.csv
/kaggle/input/march-machine-learning-mania-2026/WRegularSeasonCompactResults.csv
/kaggle/input/march-machine-learning-mania-2026/MNCAATourneySeedRoundSlots.csv
/kaggle/input/march-machine-learning-mania-2026/MRegularSeasonDetailedResults.csv
/kaggle/input/march-machine-learning-mania-2026/MNCAATourneyCompactResults.csv
/kaggle/input/march-machine-learning-mania-2026/MGameCities.csv
/kaggle/input/march-machine-learning-mania-2026/WSecondaryTourneyCompactResults.csv
/kaggle/input/march-machine-learning-mania-2026/WGameCities.csv
/kaggle/input/march-machine-learning-mania-2026/MSeasons.csv
/kaggle/input/march-machine-learning-mania-2026/WNCAATourneySlots.csv
/kaggle/input/march-machine-learning-mania-2026/MSecondaryTourneyTeams.csv
/kaggle/input/march-machine-learning-mania-2026/SampleSubmissionStage2.csv
/kaggle/input/march-machine-learning-mania

In [2]:
pip install lightgbm scikit-learn pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import brier_score_loss
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb

In [4]:
# Kaggle data files live here — do not change this
DATA_DIR = "/kaggle/input/march-machine-learning-mania-2026"

In [5]:
print("Loading all files...")
 
# --- Shared (no M/W prefix) ---
cities      = pd.read_csv(f"{DATA_DIR}/Cities.csv")
conferences = pd.read_csv(f"{DATA_DIR}/Conferences.csv")
 
# Men's 
m_teams            = pd.read_csv(f"{DATA_DIR}/MTeams.csv")
m_seasons          = pd.read_csv(f"{DATA_DIR}/MSeasons.csv")
m_team_spellings   = pd.read_csv(f"{DATA_DIR}/MTeamSpellings.csv")
m_team_conf        = pd.read_csv(f"{DATA_DIR}/MTeamConferences.csv")
m_team_coaches     = pd.read_csv(f"{DATA_DIR}/MTeamCoaches.csv")
m_reg_compact      = pd.read_csv(f"{DATA_DIR}/MRegularSeasonCompactResults.csv")
m_reg_detailed     = pd.read_csv(f"{DATA_DIR}/MRegularSeasonDetailedResults.csv")
m_tourney_compact  = pd.read_csv(f"{DATA_DIR}/MNCAATourneyCompactResults.csv")
m_tourney_detailed = pd.read_csv(f"{DATA_DIR}/MNCAATourneyDetailedResults.csv")
m_seeds            = pd.read_csv(f"{DATA_DIR}/MNCAATourneySeeds.csv")
m_slots            = pd.read_csv(f"{DATA_DIR}/MNCAATourneySlots.csv")
m_seed_slots       = pd.read_csv(f"{DATA_DIR}/MNCAATourneySeedRoundSlots.csv")
m_massey           = pd.read_csv(f"{DATA_DIR}/MMasseyOrdinals.csv")
m_game_cities      = pd.read_csv(f"{DATA_DIR}/MGameCities.csv")
m_conf_tourney     = pd.read_csv(f"{DATA_DIR}/MConferenceTourneyGames.csv")
m_secondary_teams  = pd.read_csv(f"{DATA_DIR}/MSecondaryTourneyTeams.csv")
m_secondary_res    = pd.read_csv(f"{DATA_DIR}/MSecondaryTourneyCompactResults.csv")
 
# Women's 
w_teams            = pd.read_csv(f"{DATA_DIR}/WTeams.csv")
w_seasons          = pd.read_csv(f"{DATA_DIR}/WSeasons.csv")
w_team_spellings   = pd.read_csv(f"{DATA_DIR}/WTeamSpellings.csv")
w_team_conf        = pd.read_csv(f"{DATA_DIR}/WTeamConferences.csv")
w_reg_compact      = pd.read_csv(f"{DATA_DIR}/WRegularSeasonCompactResults.csv")
w_reg_detailed     = pd.read_csv(f"{DATA_DIR}/WRegularSeasonDetailedResults.csv")
w_tourney_compact  = pd.read_csv(f"{DATA_DIR}/WNCAATourneyCompactResults.csv")
w_tourney_detailed = pd.read_csv(f"{DATA_DIR}/WNCAATourneyDetailedResults.csv")
w_seeds            = pd.read_csv(f"{DATA_DIR}/WNCAATourneySeeds.csv")
w_slots            = pd.read_csv(f"{DATA_DIR}/WNCAATourneySlots.csv")
w_game_cities      = pd.read_csv(f"{DATA_DIR}/WGameCities.csv")
w_conf_tourney     = pd.read_csv(f"{DATA_DIR}/WConferenceTourneyGames.csv")
w_secondary_teams  = pd.read_csv(f"{DATA_DIR}/WSecondaryTourneyTeams.csv")
w_secondary_res    = pd.read_csv(f"{DATA_DIR}/WSecondaryTourneyCompactResults.csv")
 
# Submission 
sub1 = pd.read_csv(f"{DATA_DIR}/SampleSubmissionStage1.csv")
sub2 = pd.read_csv(f"{DATA_DIR}/SampleSubmissionStage2.csv")
 
print(f"Stage 1 template : {len(sub1):>8,} rows")
print(f"Stage 2 template : {len(sub2):>8,} rows")
print("All files loaded ")
 

Loading all files...
Stage 1 template :  519,144 rows
Stage 2 template :  132,133 rows
All files loaded 


 FEATURE FUNCTIONS
##### Each function takes raw DataFrames and returns a clean
##### (Season, TeamID, feature...) table ready for merging.

In [6]:
# Feature 1: Box-score averages
# Source: MRegularSeasonDetailedResults / WRegularSeasonDetailedResults
# Each game has W-side and L-side stats. We stack both,
# then average per team per season.
# Gives us: avg scoring, shooting %, rebounding, assists, turnovers etc.
 
def compute_box_score_stats(reg_detailed):
    raw = ["FGM","FGA","FGM3","FGA3","FTM","FTA",
           "OR","DR","Ast","TO","Stl","Blk","PF","Score"]
 
    def one_side(df, side):
        rename = {f"{side}{c}": c for c in raw}
        rename[f"{side}TeamID"] = "TeamID"
        return df[["Season"] + list(rename.keys())].rename(columns=rename)
 
    games = pd.concat([one_side(reg_detailed, "W"),
                       one_side(reg_detailed, "L")], ignore_index=True)
 
    games["FGpct"]  = games["FGM"]  / games["FGA"].replace(0, np.nan)
    games["FG3pct"] = games["FGM3"] / games["FGA3"].replace(0, np.nan)
    games["FTpct"]  = games["FTM"]  / games["FTA"].replace(0, np.nan)
    games["TRB"]    = games["OR"] + games["DR"]
 
    cols = ["Score","FGpct","FG3pct","FTpct","TRB","Ast","TO","Stl","Blk","PF"]
    return (games.groupby(["Season","TeamID"])[cols]
                 .mean().reset_index()
                 .rename(columns={c: f"avg_{c}" for c in cols}))
 

In [7]:
# Feature 2: Win rate + point differential 
# Source: MRegularSeasonCompactResults / WRegularSeasonCompactResults
# win_rate       = wins / total games
# avg_point_diff = average score margin across all games
#                  (positive = team wins by more than it loses by)
 
def compute_win_rates(reg_compact):
    wins = (reg_compact.groupby(["Season","WTeamID"]).size()
            .reset_index(name="wins").rename(columns={"WTeamID":"TeamID"}))
    losses = (reg_compact.groupby(["Season","LTeamID"]).size()
              .reset_index(name="losses").rename(columns={"LTeamID":"TeamID"}))
    rec = wins.merge(losses, on=["Season","TeamID"], how="outer").fillna(0)
    rec["games"]    = rec["wins"] + rec["losses"]
    rec["win_rate"] = rec["wins"] / rec["games"].replace(0, np.nan)
 
    wm = (reg_compact.groupby(["Season","WTeamID"])
          .apply(lambda x: (x["WScore"] - x["LScore"]).mean())
          .reset_index(name="wm").rename(columns={"WTeamID":"TeamID"}))
    lm = (reg_compact.groupby(["Season","LTeamID"])
          .apply(lambda x: (x["LScore"] - x["WScore"]).mean())
          .reset_index(name="lm").rename(columns={"LTeamID":"TeamID"}))
 
    rec = rec.merge(wm, on=["Season","TeamID"], how="left")
    rec = rec.merge(lm, on=["Season","TeamID"], how="left")
    rec["avg_point_diff"] = (
        rec["wm"].fillna(0) * rec["wins"] +
        rec["lm"].fillna(0) * rec["losses"]
    ) / rec["games"].replace(0, np.nan)
 
    return rec[["Season","TeamID","win_rate","avg_point_diff"]]
 
 

In [8]:
#  Feature 3: Recent form — last 10 regular season games
# Source: MRegularSeasonCompactResults / WRegularSeasonCompactResults
# Teams that are hot heading into the tournament do better.
# DayNum lets us sort games chronologically within a season.
 
def compute_recent_form(reg_compact, last_n=10):
    w = reg_compact[["Season","DayNum","WTeamID"]].copy()
    w["TeamID"] = w["WTeamID"]; w["won"] = 1
    l = reg_compact[["Season","DayNum","LTeamID"]].copy()
    l["TeamID"] = l["LTeamID"]; l["won"] = 0
 
    all_g = pd.concat([w[["Season","DayNum","TeamID","won"]],
                        l[["Season","DayNum","TeamID","won"]]],
                       ignore_index=True)
    all_g = all_g.sort_values(["Season","TeamID","DayNum"])
 
    return (all_g.groupby(["Season","TeamID"])
                 .tail(last_n)
                 .groupby(["Season","TeamID"])["won"]
                 .mean().reset_index()
                 .rename(columns={"won": "last10_win_rate"}))
 
 

In [9]:
# Feature 4: Tournament seed 
# Source: MNCAATourneySeeds / WNCAATourneySeeds
# Seeds look like "W01", "X12", "Z16a" — extract just the number.
# 1 = top seed (best), 16 = bottom seed (worst).
 
def get_seed_map(seeds_df):
    s = seeds_df.copy()
    s["SeedNum"] = s["Seed"].str.extract(r"(\d+)").astype(int)
    return s.set_index(["Season","TeamID"])["SeedNum"].to_dict()
 

In [10]:
#  Feature 5: Massey Ordinals 
# Source: MMasseyOrdinals (Men's only — no women's file)
# ~40 external ranking systems (Pomeroy, Sagarin, RPI, ESPN…)
# We use the last snapshot before the tournament: RankingDayNum=133
# Summarised into: average rank and best rank across all systems.
# Lower rank number = better team.
 
def compute_massey(massey_df):
    pre = massey_df[massey_df["RankingDayNum"] == 133]
    return (pre.groupby(["Season","TeamID"])["OrdinalRank"]
               .agg(massey_avg="mean", massey_best="min")
               .reset_index())
 
 

In [11]:
# Feature 6: Conference strength
# Source: MTeamConferences / WTeamConferences +
#         MNCAATourneyCompactResults / WNCAATourneyCompactResults
# A team from a strong conference (e.g. ACC, Big Ten) tends to
# be better prepared than one from a weak conference.
# We measure conference strength as its historical tournament
# win rate across all seasons.
 
def compute_conf_strength(tourney_compact, team_conf):
    tc = team_conf[["Season","TeamID","ConfAbbrev"]]
 
    w = tourney_compact[["Season","WTeamID"]].merge(
        tc.rename(columns={"TeamID":"WTeamID","ConfAbbrev":"Conf"}),
        on=["Season","WTeamID"], how="left")
    l = tourney_compact[["Season","LTeamID"]].merge(
        tc.rename(columns={"TeamID":"LTeamID","ConfAbbrev":"Conf"}),
        on=["Season","LTeamID"], how="left")
 
    wc = w.groupby(["Season","Conf"]).size().reset_index(name="cw")
    lc = l.groupby(["Season","Conf"]).size().reset_index(name="cl")
 
    cs = wc.merge(lc, on=["Season","Conf"], how="outer").fillna(0)
    cs["conf_games"]    = cs["cw"] + cs["cl"]
    cs["conf_win_rate"] = cs["cw"] / cs["conf_games"].replace(0, np.nan)
    cs = cs.rename(columns={"Conf":"ConfAbbrev"})
 
    # Merge back onto teams
    result = tc.merge(cs[["Season","ConfAbbrev","conf_win_rate"]],
                      on=["Season","ConfAbbrev"], how="left")
    return result[["Season","TeamID","conf_win_rate"]]
 
 

In [12]:
#  Feature 7: Conference tournament performance 
# Source: MConferenceTourneyGames / WConferenceTourneyGames
# How many conf tournament wins a team had in that season.
# Conference tournaments happen right before March Madness
# so they reflect current-season form very well.
 
def compute_conf_tourney_perf(conf_tourney):
    wins = (conf_tourney.groupby(["Season","WTeamID"]).size()
            .reset_index(name="ctourney_wins")
            .rename(columns={"WTeamID":"TeamID"}))
    losses = (conf_tourney.groupby(["Season","LTeamID"]).size()
              .reset_index(name="ctourney_losses")
              .rename(columns={"LTeamID":"TeamID"}))
    perf = wins.merge(losses, on=["Season","TeamID"], how="outer").fillna(0)
    perf["ctourney_games"]    = perf["ctourney_wins"] + perf["ctourney_losses"]
    perf["ctourney_win_rate"] = (
        perf["ctourney_wins"] / perf["ctourney_games"].replace(0, np.nan)
    )
    return perf[["Season","TeamID","ctourney_wins","ctourney_win_rate"]]
 

In [13]:
# Feature 8: Coach tournament experience 
# Source: MTeamCoaches (Men's only — no women's coaches file)
# Coaches who have guided more NCAA tournament runs tend to
# perform better. We count how many prior seasons each coach
# appeared in the tournament before the current season.
 
def compute_coach_experience(tourney_compact, team_coaches):
    # Get teams that appeared in each tournament
    appeared = pd.concat([
        tourney_compact[["Season","WTeamID"]].rename(columns={"WTeamID":"TeamID"}),
        tourney_compact[["Season","LTeamID"]].rename(columns={"LTeamID":"TeamID"}),
    ]).drop_duplicates()
 
    # Match the coach active at tournament time (around DayNum 132)
    coach_at = team_coaches[
        (team_coaches["FirstDayNum"] <= 132) &
        (team_coaches["LastDayNum"]  >= 132)
    ][["Season","TeamID","CoachName"]]
 
    appeared = appeared.merge(coach_at, on=["Season","TeamID"], how="left")
    appeared = appeared.sort_values("Season")
 
    # cumcount gives 0 for first appearance, 1 for second, etc.
    appeared["coach_tourney_exp"] = appeared.groupby("CoachName").cumcount()
 
    return appeared[["Season","TeamID","coach_tourney_exp"]]
 

In [14]:
# Feature 9: Neutral court experience 
# Source: MRegularSeasonCompactResults / WRegularSeasonCompactResults
# The entire NCAA tournament is played on neutral courts.
# Teams that play more neutral-site regular season games
# may be more comfortable in that environment.
# WLoc = "N" means the game was on a neutral court.
 
def compute_neutral_experience(reg_compact):
    w = reg_compact[["Season","WTeamID","WLoc"]].copy()
    w["TeamID"] = w["WTeamID"]
    w["neutral"] = (w["WLoc"] == "N").astype(int)
 
    l = reg_compact[["Season","LTeamID","WLoc"]].copy()
    l["TeamID"] = l["LTeamID"]
    l["neutral"] = (l["WLoc"] == "N").astype(int)
 
    both = pd.concat([w[["Season","TeamID","neutral"]],
                       l[["Season","TeamID","neutral"]]], ignore_index=True)
    return (both.groupby(["Season","TeamID"])["neutral"]
                .mean().reset_index()
                .rename(columns={"neutral":"neutral_rate"}))
 

In [15]:
#  Feature 10: Game location geography
# Source: MGameCities / WGameCities + Cities.csv
# We count how many of a team's regular season games were
# played in their home state vs away. Teams that travel more
# may be more accustomed to road conditions.
 
def compute_travel_experience(game_cities, cities, reg_compact):
    # Attach state info to each game
    gc = game_cities.merge(cities[["CityID","State"]], on="CityID", how="left")
 
    # Only regular season games
    gc_reg = gc[gc["CRType"] == "Regular"][
        ["Season","WTeamID","LTeamID","State"]
    ]
 
    w = gc_reg[["Season","WTeamID","State"]].rename(columns={"WTeamID":"TeamID"})
    l = gc_reg[["Season","LTeamID","State"]].rename(columns={"LTeamID":"TeamID"})
    all_g = pd.concat([w, l], ignore_index=True)
 
    total_games = (all_g.groupby(["Season","TeamID"]).size()
                        .reset_index(name="total_games"))
    unique_states = (all_g.groupby(["Season","TeamID"])["State"]
                          .nunique().reset_index(name="states_played_in"))
 
    result = total_games.merge(unique_states, on=["Season","TeamID"])
    result["travel_diversity"] = result["states_played_in"] / result["total_games"]
    return result[["Season","TeamID","travel_diversity"]]
 
 

In [16]:
# Feature 11: Secondary tournament performance 
# Source: MSecondaryTourneyCompactResults / WSecondaryTourneyCompactResults
#         MSecondaryTourneyTeams / WSecondaryTourneyTeams
# Teams that appear in secondary tournaments (NIT, CBI etc.)
# in prior seasons but then make the NCAA tournament later
# show upward trajectory. We count prior NIT/secondary appearances.
 
def compute_secondary_experience(secondary_teams):
    secondary_teams = secondary_teams.sort_values("Season")
    exp = (secondary_teams.groupby("TeamID")
           .apply(lambda x: x.assign(
               prior_secondary=range(len(x))
           )).reset_index(drop=True))
    return exp[["Season","TeamID","prior_secondary"]]
 
 

In [17]:
#  Feature 12: Tournament box score history 
# Source: MNCAATourneyDetailedResults / WNCAATourneyDetailedResults
# How a team historically performs IN the tournament
# (not just regular season). Tournament games are higher pressure
# so this is a distinct and valuable signal.
 
def compute_tourney_box_history(tourney_detailed):
    raw = ["FGM","FGA","FGM3","FGA3","FTM","FTA",
           "OR","DR","Ast","TO","Stl","Blk","PF","Score"]
 
    def one_side(df, side):
        rename = {f"{side}{c}": c for c in raw}
        rename[f"{side}TeamID"] = "TeamID"
        return df[["Season"] + list(rename.keys())].rename(columns=rename)
 
    games = pd.concat([one_side(tourney_detailed, "W"),
                       one_side(tourney_detailed, "L")], ignore_index=True)
 
    games["FGpct"]  = games["FGM"]  / games["FGA"].replace(0, np.nan)
    games["FG3pct"] = games["FGM3"] / games["FGA3"].replace(0, np.nan)
    games["FTpct"]  = games["FTM"]  / games["FTA"].replace(0, np.nan)
    games["TRB"]    = games["OR"] + games["DR"]
 
    # Use all PRIOR seasons' tournament data (not current season —
    # we don't know current season results yet)
    # We do a cumulative average up to but not including current season
    cols = ["Score","FGpct","FG3pct","FTpct","TRB","Ast","TO","Stl","Blk","PF"]
    season_avg = (games.groupby(["Season","TeamID"])[cols]
                       .mean().reset_index())
 
    # For each season, average all PRIOR seasons
    records = []
    for season in sorted(season_avg["Season"].unique()):
        prior = season_avg[season_avg["Season"] < season]
        if prior.empty:
            continue
        avg = (prior.groupby("TeamID")[cols]
                    .mean().reset_index()
                    .rename(columns={c: f"hist_tourney_{c}" for c in cols}))
        avg["Season"] = season
        records.append(avg)
 
    if not records:
        return pd.DataFrame(columns=["Season","TeamID"])
 
    return pd.concat(records, ignore_index=True)
 
 

COMPUTE ALL FEATURES FOR MEN'S AND WOMEN'S

In [18]:
print("\nComputing Men's features...")
m_box_stats    = compute_box_score_stats(m_reg_detailed)
m_win_rates    = compute_win_rates(m_reg_compact)
m_recent_form  = compute_recent_form(m_reg_compact)
m_seed_map     = get_seed_map(m_seeds)
m_massey_feat  = compute_massey(m_massey)
m_conf_str     = compute_conf_strength(m_tourney_compact, m_team_conf)
m_conf_perf    = compute_conf_tourney_perf(m_conf_tourney)
m_coach_exp    = compute_coach_experience(m_tourney_compact, m_team_coaches)
m_neutral      = compute_neutral_experience(m_reg_compact)
m_travel       = compute_travel_experience(m_game_cities, cities, m_reg_compact)
m_secondary    = compute_secondary_experience(m_secondary_teams)
m_hist_tourney = compute_tourney_box_history(m_tourney_detailed)
print("  Men's features done ")
 
print("Computing Women's features...")
w_box_stats    = compute_box_score_stats(w_reg_detailed)
w_win_rates    = compute_win_rates(w_reg_compact)
w_recent_form  = compute_recent_form(w_reg_compact)
w_seed_map     = get_seed_map(w_seeds)
w_conf_str     = compute_conf_strength(w_tourney_compact, w_team_conf)
w_conf_perf    = compute_conf_tourney_perf(w_conf_tourney)
w_neutral      = compute_neutral_experience(w_reg_compact)
w_travel       = compute_travel_experience(w_game_cities, cities, w_reg_compact)
w_secondary    = compute_secondary_experience(w_secondary_teams)
w_hist_tourney = compute_tourney_box_history(w_tourney_detailed)
print("  Women's features done ")
 
 


Computing Men's features...
  Men's features done 
Computing Women's features...
  Women's features done 


BUILD MATCHUP FEATURE ROWS

In [19]:
def merge_for_team(df, team_col, feature_df, feature_cols):

    # Rename TeamID to match the team column name in df
    fdf = feature_df[["Season","TeamID"] + feature_cols].rename(
        columns={"TeamID": team_col}
    )
    # Rename feature columns to T1_* or T2_*
    fdf = fdf.rename(columns={c: f"{team_col}_{c}" for c in feature_cols})
    return df.merge(fdf, on=["Season", team_col], how="left")
 
 
def build_matchup_features(pairs,
                            box_stats, win_rates, recent_form,
                            seed_map, massey_feat,
                            conf_str, conf_perf,
                            coach_exp, neutral, travel,
                            secondary, hist_tourney):
    df = pairs.copy()
 
    # Seeds
    df["T1_seed"]  = df.apply(lambda r: seed_map.get((r["Season"], r["T1"]), 16), axis=1)
    df["T2_seed"]  = df.apply(lambda r: seed_map.get((r["Season"], r["T2"]), 16), axis=1)
    df["seed_diff"] = df["T1_seed"] - df["T2_seed"]
 
    # Box score stats
    box_cols = [c for c in box_stats.columns if c.startswith("avg_")]
    for t in ["T1","T2"]:
        df = merge_for_team(df, t, box_stats, box_cols)
    for c in box_cols:
        df[f"diff_{c}"] = df[f"T1_{c}"] - df[f"T2_{c}"]
 
    # Win rate + point differential
    for t in ["T1","T2"]:
        df = merge_for_team(df, t, win_rates, ["win_rate","avg_point_diff"])
    df["diff_win_rate"]   = df["T1_win_rate"]    - df["T2_win_rate"]
    df["diff_point_diff"] = df["T1_avg_point_diff"] - df["T2_avg_point_diff"]
 
    # Recent form
    for t in ["T1","T2"]:
        df = merge_for_team(df, t, recent_form, ["last10_win_rate"])
    df["diff_last10"] = df["T1_last10_win_rate"] - df["T2_last10_win_rate"]
 
    # Massey (Men's only)
    if massey_feat is not None:
        for t in ["T1","T2"]:
            df = merge_for_team(df, t, massey_feat, ["massey_avg","massey_best"])
        df["diff_massey_avg"]  = df["T1_massey_avg"]  - df["T2_massey_avg"]
        df["diff_massey_best"] = df["T1_massey_best"] - df["T2_massey_best"]
 
    # Conference strength
    for t in ["T1","T2"]:
        df = merge_for_team(df, t, conf_str, ["conf_win_rate"])
    df["diff_conf_str"] = df["T1_conf_win_rate"] - df["T2_conf_win_rate"]
 
    # Conference tournament performance
    for t in ["T1","T2"]:
        df = merge_for_team(df, t, conf_perf, ["ctourney_wins","ctourney_win_rate"])
    df["diff_ctourney_wins"]     = df["T1_ctourney_wins"]     - df["T2_ctourney_wins"]
    df["diff_ctourney_win_rate"] = df["T1_ctourney_win_rate"] - df["T2_ctourney_win_rate"]
 
    # Coach experience (Men's only)
    if coach_exp is not None:
        for t in ["T1","T2"]:
            df = merge_for_team(df, t, coach_exp, ["coach_tourney_exp"])
        df["diff_coach_exp"] = df["T1_coach_tourney_exp"] - df["T2_coach_tourney_exp"]
 
    # Neutral court experience
    for t in ["T1","T2"]:
        df = merge_for_team(df, t, neutral, ["neutral_rate"])
    df["diff_neutral"] = df["T1_neutral_rate"] - df["T2_neutral_rate"]
 
    # Travel diversity
    for t in ["T1","T2"]:
        df = merge_for_team(df, t, travel, ["travel_diversity"])
    df["diff_travel"] = df["T1_travel_diversity"] - df["T2_travel_diversity"]
 
    # Secondary tournament experience
    for t in ["T1","T2"]:
        df = merge_for_team(df, t, secondary, ["prior_secondary"])
    df["diff_secondary"] = df["T1_prior_secondary"] - df["T2_prior_secondary"]
 
    # Historical tournament box score
    if hist_tourney is not None and len(hist_tourney) > 0:
        hist_cols = [c for c in hist_tourney.columns
                     if c.startswith("hist_") and c != "Season"]
        if hist_cols:
            for t in ["T1","T2"]:
                df = merge_for_team(df, t, hist_tourney, hist_cols)
            for c in hist_cols:
                df[f"diff_{c}"] = df[f"T1_{c}"] - df[f"T2_{c}"]
 
    return df
 
 
def get_feature_cols(df):
    """All numeric columns — no IDs, no labels."""
    exclude = {"Season","T1","T2","label","ID"}
    return [c for c in df.columns
            if c not in exclude and df[c].dtype != object]
 
 

TRAIN MODEL

In [20]:
def run_training(tourney_compact,
                 box_stats, win_rates, recent_form,
                 seed_map, massey_feat,
                 conf_str, conf_perf,
                 coach_exp, neutral, travel,
                 secondary, hist_tourney,
                 gender_label):
 
    # Build labelled pairs: T1 = lower TeamID, T2 = higher TeamID
    rows = []
    for _, g in tourney_compact.iterrows():
        t1, t2 = sorted([int(g["WTeamID"]), int(g["LTeamID"])])
        rows.append({"Season": int(g["Season"]), "T1": t1, "T2": t2,
                     "label": 1 if int(g["WTeamID"]) == t1 else 0})
 
    train_df = build_matchup_features(
        pd.DataFrame(rows),
        box_stats, win_rates, recent_form,
        seed_map, massey_feat,
        conf_str, conf_perf,
        coach_exp, neutral, travel,
        secondary, hist_tourney
    )
 
    feat_cols = get_feature_cols(train_df)
 
    # Fill NaN with sensible defaults instead of dropping rows.
    # Many features are legitimately missing for some teams:
    #   - ctourney_win_rate: teams that never won a conf tourney game
    #   - prior_secondary: teams that never played in NIT etc.
    #   - hist_tourney cols: early seasons with no prior tournament data
    #   - massey: not all teams are ranked by every system
    # Dropping rows would wipe almost everything. Instead we fill:
    #   - win rates / percentages → 0.5 (neutral, no information)
    #   - count columns (wins, experience) → 0 (none recorded)
    #   - diff columns → 0 (assume no difference)
    #   - ranking columns → median rank (neutral assumption)
    for c in feat_cols:
        if train_df[c].isnull().any():
            if "win_rate" in c or "pct" in c or "rate" in c:
                train_df[c] = train_df[c].fillna(0.5)
            elif "rank" in c or "massey" in c:
                train_df[c] = train_df[c].fillna(train_df[c].median())
            else:
                train_df[c] = train_df[c].fillna(0)
 
    print(f"\n  {gender_label}: {len(train_df)} training games | {len(feat_cols)} features")
    print(f"  Remaining NaNs after fillna: {train_df[feat_cols].isnull().sum().sum()}")
 
    X, y = train_df[feat_cols].values, train_df["label"].values
 
    params = {
        "objective": "binary", "metric": "binary_logloss",
        "learning_rate": 0.05, "num_leaves": 31,
        "min_data_in_leaf": 15, "feature_fraction": 0.8,
        "bagging_fraction": 0.8, "bagging_freq": 5,
        "verbose": -1, "seed": 42,
    }
 
    models, oof = [], np.zeros(len(y))
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
 
    print(f"  {'Fold':>4}  {'Brier':>8}")
    for fold, (tr, val) in enumerate(kf.split(X, y), 1):
        m = lgb.train(
            params,
            lgb.Dataset(X[tr], y[tr]),
            num_boost_round=600,
            valid_sets=[lgb.Dataset(X[val], y[val])],
            callbacks=[lgb.early_stopping(50, verbose=False),
                       lgb.log_evaluation(0)],
        )
        oof[val] = m.predict(X[val])
        print(f"  {fold:>4}  {brier_score_loss(y[val], oof[val]):>8.4f}")
        models.append(m)
 
    print(f"  OOF Brier : {brier_score_loss(y, oof):.4f}  (baseline 0.2500)")
    return models, feat_cols
 
 
print("\n" + "="*55)
print("Training Men's model...")
m_models, m_feat_cols = run_training(
    m_tourney_compact,
    m_box_stats, m_win_rates, m_recent_form,
    m_seed_map, m_massey_feat,
    m_conf_str, m_conf_perf,
    m_coach_exp, m_neutral, m_travel,
    m_secondary, m_hist_tourney,
    "Men's"
)
 
print("\nTraining Women's model...")
w_models, w_feat_cols = run_training(
    w_tourney_compact,
    w_box_stats, w_win_rates, w_recent_form,
    w_seed_map, None,
    w_conf_str, w_conf_perf,
    None, w_neutral, w_travel,
    w_secondary, w_hist_tourney,
    "Women's"
)
 
 


Training Men's model...

  Men's: 2585 training games | 99 features
  Remaining NaNs after fillna: 0
  Fold     Brier
     1    0.1357
     2    0.1548
     3    0.1521
     4    0.1274
     5    0.1341
  OOF Brier : 0.1408  (baseline 0.2500)

Training Women's model...

  Women's: 1717 training games | 90 features
  Remaining NaNs after fillna: 0
  Fold     Brier
     1    0.1167
     2    0.1383
     3    0.1179
     4    0.1119
     5    0.1155
  OOF Brier : 0.1201  (baseline 0.2500)


GENERATE STAGE 1 AND STAGE 2 SUBMISSIONS

In [21]:
def predict_pairs(pairs, models, feat_cols,
                  box_stats, win_rates, recent_form,
                  seed_map, massey_feat,
                  conf_str, conf_perf,
                  coach_exp, neutral, travel,
                  secondary, hist_tourney):
    df    = build_matchup_features(pairs, box_stats, win_rates, recent_form,
                                    seed_map, massey_feat, conf_str, conf_perf,
                                    coach_exp, neutral, travel, secondary, hist_tourney)
    X     = df.reindex(columns=feat_cols, fill_value=0).values
    preds = np.mean([m.predict(X) for m in models], axis=0)
    return np.clip(preds, 0.025, 0.975)
 
 
def make_submission(sub_template):
    parts = sub_template["ID"].str.split("_", expand=True)
    df = sub_template.copy()
    df["Season"] = parts[0].astype(int)
    df["T1"]     = parts[1].astype(int)
    df["T2"]     = parts[2].astype(int)
 
    men_mask   = df["T1"].between(1000, 1999)
    women_mask = df["T1"].between(3000, 3999)
 
    print(f"  Men's rows   : {men_mask.sum():>8,}")
    print(f"  Women's rows : {women_mask.sum():>8,}")
    print(f"  Total        : {len(df):>8,}")
 
    result = sub_template[["ID"]].copy()
    result["Pred"] = 0.5
 
    if men_mask.sum() > 0:
        result.loc[men_mask.values, "Pred"] = predict_pairs(
            df[men_mask][["Season","T1","T2"]].reset_index(drop=True),
            m_models, m_feat_cols,
            m_box_stats, m_win_rates, m_recent_form,
            m_seed_map, m_massey_feat,
            m_conf_str, m_conf_perf,
            m_coach_exp, m_neutral, m_travel,
            m_secondary, m_hist_tourney
        )
 
    if women_mask.sum() > 0:
        result.loc[women_mask.values, "Pred"] = predict_pairs(
            df[women_mask][["Season","T1","T2"]].reset_index(drop=True),
            w_models, w_feat_cols,
            w_box_stats, w_win_rates, w_recent_form,
            w_seed_map, None,
            w_conf_str, w_conf_perf,
            None, w_neutral, w_travel,
            w_secondary, w_hist_tourney
        )
 
    return result
 
 
print("\n" + "="*55)
print("Generating Stage 1 submission (seasons 2021–2025)...")
stage1 = make_submission(sub1)
stage1.to_csv("submission_stage1.csv", index=False)
print(f"  Saved: submission_stage1.csv — {len(stage1):,} rows")
 
print("\nGenerating Stage 2 submission (season 2026)...")
stage2 = make_submission(sub2)
stage2.to_csv("submission_stage2.csv", index=False)
print(f"  Saved: submission_stage2.csv — {len(stage2):,} rows")

print("  Submit submission_stage1.csv → Stage 1")
print("  Submit submission_stage2.csv → Stage 2 (your real entry)")
print("  MANUALLY SELECT Stage 2 on Kaggle before the Thursday deadline")
 


Generating Stage 1 submission (seasons 2021–2025)...
  Men's rows   :  261,013
  Women's rows :  258,131
  Total        :  519,144
  Saved: submission_stage1.csv — 519,144 rows

Generating Stage 2 submission (season 2026)...
  Men's rows   :   66,430
  Women's rows :   65,703
  Total        :  132,133
  Saved: submission_stage2.csv — 132,133 rows
  Submit submission_stage1.csv → Stage 1
  Submit submission_stage2.csv → Stage 2 (your real entry)
  MANUALLY SELECT Stage 2 on Kaggle before the Thursday deadline
